# Logs

> Generating / formatting the base logging instance

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Annotated
from enum import Enum

import os, sys, json
from pathlib import Path

import typer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

from mcp.server.fastmcp import FastMCP

from nbdev_mcp import __version__
from nbdev_mcp.utils.logs import log
from nbdev_mcp.utils.config import CURRENT_PROJECT
from nbdev_mcp.utils.paths import resolve_selector
from nbdev_mcp.utils.subprocess import watch_notebooks
from nbdev_mcp.resources import add_resources
from nbdev_mcp.tools import (
    add_project_tools, add_env_tools, add_nbdev_tools, 
    add_notebook_editing_tools, add_lint_tools, 
    add_analysis_tools, add_tests_tools
)
from nbdev_mcp.prompts import add_prompts
from nbdev_mcp.tasks import add_task_tools, enable_nbdev_tasks
from nbdev_mcp.tests.agent_e2e import add_recording_tools, enable_auto_recording

console = Console();  # semicolon suppresses notebook output

## HTTP Utils

In [ ]:
#| export
def set_http_path_if_supported(target_path: str) -> bool:
    """Try to set the HTTP mount path on the MCP SDK settings.
    
    Parameters
    ----------
    target_path : str
        The path to set (e.g., '/mcp').
    
    Returns
    -------
    bool
        True if successfully set, False if not supported.
    """
    import mcp
    try:
        mcp.settings.streamable_http_path = target_path  # type: ignore[attr-defined]
        return True
    except Exception:
        try:
            mcp.settings.http_path = target_path  # type: ignore[attr-defined]
            return True
        except Exception:
            return False

## Create the MCP

In [ ]:
#| export
def create_nbdev_mcp(name: str = "mcp.nbdev") -> FastMCP:
    """Create and configure the nbdev MCP server with all resources, tools, and prompts."""
    mcp = FastMCP(name)
    
    # Enable experimental tasks API
    enable_nbdev_tasks(mcp)
    
    # Attach all nbdev-related resources, tools, and prompts
    add_resources(mcp)
    add_project_tools(mcp)
    add_env_tools(mcp)
    add_nbdev_tools(mcp)
    add_notebook_editing_tools(mcp)  # CRITICAL: Notebook editing and workflow tools
    add_prompts(mcp)  # Philosophy prompts must come after tools are registered
    
    # Extensions: linting, analysis, and code generation tools
    add_lint_tools(mcp)
    add_analysis_tools(mcp)
    add_tests_tools(mcp)
    
    # Task-based tools for auditing, deduplication, and refactoring
    add_task_tools(mcp)
    
    # E2E testing: recording tools for agent behavior analysis
    add_recording_tools(mcp)
    enable_auto_recording(mcp)  # Enable automatic recording when recording is active
    
    return mcp

## `main`

In [ ]:
#| export
class Transport(str, Enum):
    """MCP transport modes."""
    stdio = "stdio"
    http = "http"
    streamable_http = "streamable-http"


class Provider(str, Enum):
    """Supported MCP client providers."""
    claude = "claude"
    vscode = "vscode"
    cursor = "cursor"
    codex = "codex"


app = typer.Typer(
    name="nbdev-mcp",
    help="[bold blue]nbdev MCP server[/] - Model Context Protocol for nbdev projects",
    rich_markup_mode="rich",
    no_args_is_help=False,
    add_completion=True,
    context_settings={"help_option_names": ["-h", "--help"]},
);  # semicolon suppresses notebook output


def version_callback(value: bool):
    if value:
        rprint(f"[bold blue]nbdev-mcp[/] [dim]v{__version__}[/]")
        raise typer.Exit()


@app.callback(invoke_without_command=True)
def callback(
    ctx: typer.Context,
    version: Annotated[bool, typer.Option(
        "-V", "--version", callback=version_callback, is_eager=True,
        help="Show version and exit."
    )] = False,
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for an nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport mode.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Enable debug output."
    )] = False,
):
    """Run the MCP server (default command)."""
    if ctx.invoked_subcommand is None:
        _run_server(project=project, transport=transport, verbose=verbose)

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install helpers

In [ ]:
#| export
def get_claude_config_path() -> Path:
    """Get Claude Desktop config path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Claude/claude_desktop_config.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Claude/claude_desktop_config.json"
    else:
        return Path.home() / ".config/Claude/claude_desktop_config.json"


def get_vscode_config_path() -> Path:
    """Get VS Code MCP config path (mcp.json, not settings.json)."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Code/User/mcp.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Code/User/mcp.json"
    else:
        return Path.home() / ".config/Code/User/mcp.json"


def get_vscode_settings_path() -> Path:
    """Get VS Code settings.json path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Code/User/settings.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Code/User/settings.json"
    else:
        return Path.home() / ".config/Code/User/settings.json"


def get_cursor_config_path() -> Path:
    """Get Cursor MCP config path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Cursor/User/mcp.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Cursor/User/mcp.json"
    else:
        return Path.home() / ".config/Cursor/User/mcp.json"


def get_cursor_settings_path() -> Path:
    """Get Cursor settings.json path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Cursor/User/settings.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Cursor/User/settings.json"
    else:
        return Path.home() / ".config/Cursor/User/settings.json"


def get_codex_config_path() -> Path:
    """Get Codex CLI config path."""
    if sys.platform == "darwin":
        return Path.home() / ".codex/config.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("USERPROFILE", "")) / ".codex/config.json"
    else:
        return Path.home() / ".codex/config.json"


def get_config_path(provider: Provider) -> Path:
    """Get config path for a provider."""
    match provider:
        case Provider.claude:
            return get_claude_config_path()
        case Provider.vscode:
            return get_vscode_config_path()
        case Provider.cursor:
            return get_cursor_config_path()
        case Provider.codex:
            return get_codex_config_path()


def get_mcp_key(provider: Provider) -> str:
    """Get the MCP servers key for a provider's config."""
    match provider:
        case Provider.claude | Provider.codex:
            return "mcpServers"
        case Provider.vscode | Provider.cursor:
            return "servers"  # mcp.json uses "servers" not "mcpServers"


def get_python_path() -> str:
    """Get current Python interpreter path."""
    return sys.executable


def make_server_config() -> dict:
    """Generate server configuration dict."""
    return {
        "command": get_python_path(),
        "args": ["-u", "-m", "nbdev_mcp"]
    }


def make_server_config_for_provider(provider: Provider, auto_start: bool = False) -> dict:
    """Generate server configuration dict for a specific provider."""
    base = {
        "command": get_python_path(),
        "args": ["-u", "-m", "nbdev_mcp"]
    }
    # VS Code/Cursor mcp.json format includes "type" and optional "autoStart"
    if provider in (Provider.vscode, Provider.cursor):
        config = {"type": "stdio", **base}
        if auto_start:
            config["autoStart"] = True
        return config
    return base


def get_wrapper_script_path() -> Path:
    """Get path for the keep-alive wrapper script."""
    if sys.platform == "win32":
        return Path.home() / ".nbdev-mcp" / "nbdev-mcp-wrapper.bat"
    return Path.home() / ".nbdev-mcp" / "nbdev-mcp-wrapper.sh"


def generate_wrapper_script() -> str:
    """Generate a keep-alive wrapper script that monitors and restarts the MCP."""
    python_path = get_python_path()
    if sys.platform == "win32":
        return f'''@echo off
REM nbdev-mcp keep-alive wrapper
:loop
"{python_path}" -u -m nbdev_mcp %*
echo MCP exited with code %ERRORLEVEL%, restarting in 2 seconds...
timeout /t 2 /nobreak >nul
goto loop
'''
    else:
        return f'''#!/usr/bin/env bash
# nbdev-mcp keep-alive wrapper
# Monitors and restarts the MCP server if it exits

PYTHON="{python_path}"
RESTART_DELAY=2

while true; do
    echo "[nbdev-mcp] Starting server..."
    "$PYTHON" -u -m nbdev_mcp "$@"
    EXIT_CODE=$?
    echo "[nbdev-mcp] Server exited with code $EXIT_CODE, restarting in ${{RESTART_DELAY}}s..."
    sleep $RESTART_DELAY
done
'''


def install_wrapper_script(dry_run: bool = False) -> Path:
    """Install the keep-alive wrapper script."""
    script_path = get_wrapper_script_path()
    script_content = generate_wrapper_script()

    if dry_run:
        console.print(f"\n[bold cyan]Wrapper Script[/]")
        console.print(f"[dim]File:[/] {script_path}")
        console.print(f"\n[dim]Script content:[/]")
        console.print(script_content)
        return script_path

    script_path.parent.mkdir(parents=True, exist_ok=True)
    script_path.write_text(script_content)
    if sys.platform != "win32":
        script_path.chmod(0o755)
    console.print(f"[green]✓[/] Wrapper script installed: {script_path}")
    return script_path


def _parse_jsonc(text: str) -> dict:
    """Parse JSON with Comments (JSONC) - strips // and /* */ comments."""
    import re
    # Try standard JSON first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Strip JSONC comments and trailing commas
    text = re.sub(r'//[^\n]*', '', text)  # single-line comments
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)  # multi-line comments
    text = re.sub(r',(\s*[}\]])', r'\1', text)  # trailing commas
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}


def enable_mcp_autostart_in_settings(provider: Provider, dry_run: bool = False) -> bool:
    """Enable chat.mcp.autostart in VS Code/Cursor settings.json.
    
    VS Code requires this setting for MCP servers to auto-start.
    """
    if provider == Provider.vscode:
        settings_path = get_vscode_settings_path()
    elif provider == Provider.cursor:
        settings_path = get_cursor_settings_path()
    else:
        return False  # Only VS Code/Cursor need this
    
    # Read existing settings
    if settings_path.exists():
        settings = _parse_jsonc(settings_path.read_text())
    else:
        settings = {}
    
    # Check if already enabled
    if settings.get("chat.mcp.autostart") is True:
        console.print(f"[dim]  chat.mcp.autostart already enabled[/]")
        return True
    
    # Enable autostart
    settings["chat.mcp.autostart"] = True
    
    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()} Settings[/]")
        console.print(f"[dim]File:[/] {settings_path}")
        console.print(f"[dim]Action:[/] Set chat.mcp.autostart = true")
        return True
    
    settings_path.parent.mkdir(parents=True, exist_ok=True)
    settings_path.write_text(json.dumps(settings, indent=2))
    console.print(f"[green]✓[/] Enabled chat.mcp.autostart in {settings_path}")
    return True


def install_to_provider(provider: Provider, dry_run: bool = False, auto_start: bool = False) -> bool:
    """Install nbdev-mcp to a provider's config."""
    config_path = get_config_path(provider)
    server_config = make_server_config_for_provider(provider, auto_start=auto_start)
    mcp_key = get_mcp_key(provider)
    
    # Read existing or create new
    if config_path.exists():
        config = _parse_jsonc(config_path.read_text())
        existed = True
    else:
        config = {}
        existed = False
    
    # Build the change
    if mcp_key not in config:
        config[mcp_key] = {}
    config[mcp_key]["nbdev"] = server_config
    
    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()}[/]")
        console.print(f"[dim]File:[/] {config_path}")
        if existed:
            console.print(f"[dim]Action:[/] Update existing config")
        else:
            console.print(f"[dim]Action:[/] Create new config file")
        console.print(f"\n[dim]Full config that would be written:[/]")
        console.print_json(json.dumps(config, indent=2))
    else:
        config_path.parent.mkdir(parents=True, exist_ok=True)
        config_path.write_text(json.dumps(config, indent=2))
        console.print(f"[green]✓[/] Installed to [bold]{provider.value}[/]: {config_path}")
    
    # Also enable autostart in settings.json for VS Code/Cursor
    if auto_start and provider in (Provider.vscode, Provider.cursor):
        enable_mcp_autostart_in_settings(provider, dry_run=dry_run)
    
    return True


def uninstall_from_provider(provider: Provider, dry_run: bool = False) -> bool:
    """Remove nbdev-mcp from a provider's config."""
    config_path = get_config_path(provider)
    mcp_key = get_mcp_key(provider)
    
    if not config_path.exists():
        console.print(f"[yellow]⚠[/] {provider.value}: Config not found")
        return False
    
    config = _parse_jsonc(config_path.read_text())
    
    if mcp_key not in config or "nbdev" not in config.get(mcp_key, {}):
        console.print(f"[yellow]⚠[/] {provider.value}: nbdev-mcp not installed")
        return False
    
    # Remove the entry
    del config[mcp_key]["nbdev"]
    
    if dry_run:
        console.print(f"\n[bold cyan]{provider.value.title()}[/]")
        console.print(f"[dim]File:[/] {config_path}")
        console.print(f"[dim]Action:[/] Remove nbdev from {mcp_key}")
        console.print(f"\n[dim]Config after removal:[/]")
        console.print_json(json.dumps(config, indent=2))
        return True
    
    config_path.write_text(json.dumps(config, indent=2))
    console.print(f"[green]✓[/] Removed from [bold]{provider.value}[/]")
    return True


def check_provider_status(provider: Provider) -> dict:
    """Check installation status for a provider."""
    config_path = get_config_path(provider)
    mcp_key = get_mcp_key(provider)
    
    if not config_path.exists():
        return {"provider": provider.value, "installed": False, "exists": False, "path": str(config_path)}
    
    try:
        config = _parse_jsonc(config_path.read_text())
    except Exception:
        return {"provider": provider.value, "installed": False, "exists": True, "path": str(config_path), "error": "parse error"}
    
    installed = mcp_key in config and "nbdev" in config.get(mcp_key, {})
    
    # Check autostart setting for VS Code/Cursor
    autostart_enabled = None
    if provider in (Provider.vscode, Provider.cursor):
        settings_path = get_vscode_settings_path() if provider == Provider.vscode else get_cursor_settings_path()
        if settings_path.exists():
            try:
                settings = _parse_jsonc(settings_path.read_text())
                autostart_enabled = settings.get("chat.mcp.autostart", False)
            except Exception:
                pass
    
    result = {"provider": provider.value, "installed": installed, "exists": True, "path": str(config_path)}
    if autostart_enabled is not None:
        result["autostart"] = autostart_enabled
    return result

In [ ]:
#| export
def _run_server(
    project: Optional[str] = None,
    transport: Transport = Transport.stdio,
    host: str = "127.0.0.1",
    port: int = 8000,
    path: str = "/mcp",
    verbose: bool = False,
    watch: bool = False,
    watch_interval: float = 2.0,
    watch_cmd: str = "nbdev_export",
) -> None:
    """Internal: run the MCP server."""
    if verbose:
        import logging
        logging.basicConfig(level=logging.DEBUG, format="%(message)s")
        log.setLevel(logging.DEBUG)
        console.print(f"[dim]nbdev-mcp v{__version__}[/]")

    # Set project from env or arg
    proj_path = None
    project = project or os.environ.get("NBDEV_MCP_PROJECT")
    if project:
        try:
            proj_path = resolve_selector(project)
            if verbose:
                console.print(f"[green]✓[/] Project: {proj_path}")
        except Exception as e:
            console.print(f"[red]✗[/] {e}")
            if watch:
                raise typer.Exit(1)

    # Watch mode
    if watch:
        if not proj_path:
            console.print("[red]✗[/] Watch mode requires --project")
            raise typer.Exit(1)
        watch_notebooks(proj_path, interval=watch_interval, on_change=watch_cmd)
        return

    # Build MCP
    mcp = create_nbdev_mcp()
    
    # Auto-recording for agent tests
    if os.environ.get("NBDEV_MCP_AUTO_RECORD"):
        from nbdev_mcp.tests.agent_e2e import start_recording
        start_recording()
        log.info("Auto-recording enabled via NBDEV_MCP_AUTO_RECORD")
    
    # Register shutdown handler to save session
    session_file = os.environ.get("NBDEV_MCP_SESSION_FILE")
    if session_file:
        import atexit
        def save_session_on_exit():
            from nbdev_mcp.tests.agent_e2e import stop_recording
            session = stop_recording()
            if session:
                session.save(Path(session_file))
                log.info(f"Session saved to {session_file}")
        atexit.register(save_session_on_exit)
    
    match transport:
        case Transport.stdio:
            mcp.run(transport="stdio")
        case Transport.streamable_http | Transport.http:
            default = (host == "127.0.0.1" and port == 8000 and path == "/mcp")
            if default and transport == Transport.streamable_http:
                mcp.run(transport="streamable-http")
            else:
                try:
                    import uvicorn
                except ImportError:
                    console.print("[red]✗[/] uvicorn required: [dim]pip install uvicorn[/]")
                    raise typer.Exit(1)
                if path != "/mcp":
                    set_http_path_if_supported(path)
                uvicorn.run(mcp.streamable_http_app(), host=host, port=port)


@app.command()
def run(
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport: stdio, http, streamable-http.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    host: Annotated[str, typer.Option(
        "-H", "--host", help="Host for HTTP transport.",
        envvar="NBDEV_MCP_HOST"
    )] = "127.0.0.1",
    port: Annotated[int, typer.Option(
        "-P", "--port", help="Port for HTTP transport.",
        envvar="NBDEV_MCP_PORT"
    )] = 8000,
    path: Annotated[str, typer.Option(
        "--path", help="URL path for HTTP.",
        envvar="NBDEV_MCP_PATH"
    )] = "/mcp",
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Debug output."
    )] = False,
    watch: Annotated[bool, typer.Option(
        "-w", "--watch", help="Watch notebooks and auto-export."
    )] = False,
    watch_interval: Annotated[float, typer.Option(
        "--watch-interval", help="Watch polling interval (seconds)."
    )] = 2.0,
    watch_cmd: Annotated[str, typer.Option(
        "--watch-cmd", help="Command on change."
    )] = "nbdev_export",
):
    """Run the MCP server."""
    _run_server(
        project=project, transport=transport, host=host, port=port,
        path=path, verbose=verbose, watch=watch,
        watch_interval=watch_interval, watch_cmd=watch_cmd
    )

In [ ]:
#| export
@app.command()
def install(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider: claude, vscode, cursor, codex. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-d", "--dry-run", help="Show exact config changes without writing."
    )] = False,
    auto_start: Annotated[bool, typer.Option(
        "-a", "--auto-start", help="Auto-start server when VS Code/Cursor opens."
    )] = False,
    wrapper: Annotated[bool, typer.Option(
        "-w", "--wrapper", help="Install keep-alive wrapper script."
    )] = False,
):
    """Install nbdev-mcp to MCP client(s)."""
    providers = [provider] if provider else list(Provider)

    for p in providers:
        install_to_provider(p, dry_run=dry_run, auto_start=auto_start)

    if wrapper:
        install_wrapper_script(dry_run=dry_run)

    if not dry_run:
        console.print("\n[yellow]Restart your MCP client(s) to activate.[/]")


@app.command()
def uninstall(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider: claude, vscode, cursor, codex. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-d", "--dry-run", help="Show exact config changes without writing."
    )] = False,
):
    """Remove nbdev-mcp from MCP client(s)."""
    providers = [provider] if provider else list(Provider)
    
    for p in providers:
        uninstall_from_provider(p, dry_run=dry_run)


@app.command()
def status():
    """Show installation status for all providers."""
    table = Table(title="nbdev-mcp Status", show_header=True)
    table.add_column("Provider", style="cyan")
    table.add_column("Status")
    table.add_column("Auto-Start")
    table.add_column("Path", style="dim")
    
    for provider in Provider:
        info = check_provider_status(provider)
        if not info["exists"]:
            st = "[dim]Config not found[/]"
            autostart = ""
        elif info["installed"]:
            st = "[green]✓ Installed[/]"
            # Show autostart status for VS Code/Cursor
            if "autostart" in info:
                autostart = "[green]✓[/]" if info["autostart"] else "[yellow]✗[/]"
            else:
                autostart = "[dim]n/a[/]"
        else:
            st = "[yellow]Not configured[/]"
            autostart = ""
        table.add_row(provider.value.title(), st, autostart, info["path"])
    
    table.add_row("", "", "", "")
    table.add_row("Python", f"[green]v{sys.version.split()[0]}[/]", "", get_python_path())
    table.add_row("nbdev-mcp", f"[blue]v{__version__}[/]", "", "")
    
    console.print(table)

In [ ]:
#| export
@app.command()
def test(
    agent: Annotated[str, typer.Argument(
        help="Agent to test: claude or codex"
    )] = "claude",
    scenario: Annotated[Optional[str], typer.Option(
        "-s", "--scenario", help="Specific scenario to test (all if omitted)"
    )] = None,
    save_dir: Annotated[Optional[str], typer.Option(
        "-o", "--output", help="Directory to save session files"
    )] = None,
    timeout: Annotated[int, typer.Option(
        "-t", "--timeout", help="Timeout per test in seconds"
    )] = 120,
):
    """Run automated agent e2e tests.
    
    Tests agent behavior against expected patterns for MCP tool usage.
    """
    from nbdev_mcp.tests.agent_e2e import run_agent_test, run_all_agent_tests, SCENARIOS
    
    if scenario:
        if scenario not in SCENARIOS:
            console.print(f"[red]Unknown scenario: {scenario}[/]")
            console.print(f"Available: {', '.join(SCENARIOS.keys())}")
            raise typer.Exit(1)
        
        result = run_agent_test(
            agent=agent,
            scenario=scenario,
            timeout=timeout,
            save_session=Path(save_dir) / f"{scenario}.json" if save_dir else None
        )
        
        if result.get("ok"):
            console.print(f"[green]✓ PASS[/]: {scenario}")
        else:
            console.print(f"[red]✗ FAIL[/]: {scenario}")
            for failure in result.get("validation", {}).get("failures", []):
                console.print(f"  - {failure}")
    else:
        results = run_all_agent_tests(
            agent=agent,
            save_dir=Path(save_dir) if save_dir else None
        )
        
        console.print()
        table = Table(title=f"Agent Test Results ({agent})")
        table.add_column("Scenario")
        table.add_column("Status")
        table.add_column("Details")
        
        for r in results["results"]:
            status = "[green]PASS[/]" if r.get("ok") else "[red]FAIL[/]"
            details = ""
            if not r.get("ok"):
                failures = r.get("validation", {}).get("failures", [])
                details = failures[0] if failures else r.get("error", "")
            table.add_row(r.get("scenario", "?"), status, details[:50])
        
        console.print(table)
        console.print()
        console.print(f"Pass rate: {results['passed']}/{results['total']} ({results['pass_rate']:.0%})")

In [ ]:
#| export
def main():
    """Entry point for console script."""
    app()